# INTRODUÇÃO TEÓRICA: Inversão Sísmica via Redes Neurais Informadas pela Física (PINNs)

Este notebook documenta a transição da Inversão de Forma de Onda Completa (FWI) clássica para a abordagem Deep Tech utilizando Redes Neurais Informadas pela Física (PINNs). O objetivo é superar as limitações de iluminação esparsa e *Cycle Skipping* observadas nos métodos determinísticos.

## 1. A Mudança de Paradigma: FDTD vs. PINNs
No FWI clássico, o método de Diferenças Finitas no Domínio do Tempo (FDTD) é o motor central da inversão, calculando a propagação da onda de forma discreta (pixel a pixel). Nesta nova arquitetura, **o FDTD é completamente removido do loop de treinamento**. 

A Rede Neural assume o papel de aproximador universal do campo de onda. Ela aprende uma função matemática contínua $u(x, z, t)$. Para garantir que as predições da rede obedeçam às leis da física, utilizamos a Diferenciação Automática (Autograd) para extrair as derivadas exatas da rede e penalizar qualquer violação da Equação da Onda Acústica (PDE Loss). Esta formulação contínua atua como um forte regularizador espacial, permitindo que a rede extrapole a geologia para zonas de sombra (shadow zones) que o FDTD deixaria no escuro.

## 2. O Problema de Percepção: Viés Espectral e Fourier Features
Redes Neurais Densas (MLPs) sofrem de um fenômeno matemático comprovado chamado *Viés Espectral* (Spectral Bias): elas convergem rapidamente para funções de baixa frequência e têm extrema dificuldade em aprender variações abruptas. Na geofísica, uma interface rochosa (ex: salto de 1500 m/s para 3200 m/s) é um evento de alta frequência. 

Se utilizarmos uma PINN Pura, a rede preverá um campo de onda suave, zerando as derivadas espaciais e matando o gradiente geológico. Para curar esta "miopia", implementamos **Fourier Features (Positional Encoding)**. Antes de alimentar a rede, as coordenadas $(X, Z, T)$ são mapeadas para um espaço de alta dimensão através de funções trigonométricas (senos e cossenos) multiplicadas por frequências aleatórias. Isso força a rede a enxergar os detalhes finos da propagação da onda.

## 3. O Problema de Otimização: Arquitetura Híbrida (Adam -> L-BFGS)
A otimização conjunta dos pesos da rede e da matriz de velocidades cria uma topologia de erro altamente complexa.
* **Estágio 1 (ADAM):** Um otimizador de primeira ordem é utilizado para retirar os pesos da rede do estado caótico inicial. Ele é rápido e robusto, mas falha em atualizar a geologia profunda devido à atenuação geométrica do gradiente.
* **Estágio 2 (L-BFGS):** Um otimizador Quase-Newton de segunda ordem assume o controle. Ao dividir o gradiente pela aproximação da Matriz Hessiana (curvatura), o L-BFGS amplifica o sinal nas camadas profundas, esculpindo as interfaces de alta frequência que o Adam é incapaz de resolver.

A união das *Fourier Features* com a *Otimização Híbrida* forma o estado da arte atual para a resolução de problemas inversos mal postos via SciML (Scientific Machine Learning).

In [ ]:
# ==============================================================================
# CELULA 1: CONFIGURACAO DE AMBIENTE E HARDWARE (PINN)
# ==============================================================================
# DIAGRAMA DE ARQUITETURA DE HARDWARE E DEPENDENCIAS
# ------------------------------------------------------------------------------
#  [ Ambiente Virtual Isolado ] 
#          |
#          +---> [ NumPy ] ---------> Manipulação de Matrizes CPU
#          +---> [ PyTorch ] -------> Computação Tensorial, Autograd e Otimização
#          |
#          v
#  [ Alocador de Hardware: torch.device ] ---> GPU (VRAM) estritamente necessária
# ==============================================================================

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# Verificação de Hardware (Fail-Fast)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Sistema] Executando pipeline PINN. Dispositivo mapeado: {device}")
if device.type != 'cuda':
    print("[ALERTA CRITICO] CUDA não detectado. O treinamento de PINNs na CPU é inviável.")

In [ ]:
# ==============================================================================
# CELULA 2: INGESTÃO DE DADOS E ENGENHARIA DA NUVEM DE PONTOS (PINN)
# ==============================================================================
# DIAGRAMA DE TRANSFORMAÇÃO: HIPERCUBO -> NUVEM DE PONTOS CONTÍNUA
# ------------------------------------------------------------------------------
#  [ Sismograma Discreto (NumPy) ]
#  Dimensões originais: (Tiros=5, Tempo=1000, Receptores=70)
#          |
#          v ( np.meshgrid + .flatten() )
#  [ Nuvem de Pontos (PyTorch Tensors) ]
#  Coluna X: [x0, x1, x2, ..., xN] -> Coordenadas espaciais horizontais
#  Coluna Z: [z0, z1, z2, ..., zN] -> Coordenadas espaciais verticais
#  Coluna T: [t0, t1, t2, ..., tN] -> Coordenadas temporais
#          |
#          v ( Min-Max Scaling )
#  [ Domínio Topológico Normalizado ] -> Todas as coordenadas mapeadas para [-1, 1]
# ==============================================================================

import os
import numpy as np
import torch
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# ------------------------------------------------------------------------------
# 1. Definição dos Parâmetros Físicos e Geométricos
# ------------------------------------------------------------------------------
NX, NZ = 70, 70          
DX = 10.0                
DT = 0.001               
NT = 1000                
NUM_SHOTS = 5            
NUM_REC = 70             

# ------------------------------------------------------------------------------
# 2. Ingestão e Higienização de Dados (Ground Truth e Observações)
# ------------------------------------------------------------------------------
path_seismic = '../data/FlatVel_A/FlatVel_A_data14.npy'
path_model = '../data/FlatVel_A/FlatVel_A_model14.npy'

if not os.path.exists(path_seismic) or not os.path.exists(path_model):
    raise FileNotFoundError("[CRÍTICO] Arquivos do dataset OpenFWI não localizados.")

print("[Data Ingestion] Carregando tensores originais para a memória RAM...")
raw_seismic = np.load(path_seismic)
raw_model = np.load(path_model)

SAMPLE_INDEX = 0
seismic_obs = raw_seismic[SAMPLE_INDEX].copy()
true_velocity = raw_model[SAMPLE_INDEX, 0].copy()

del raw_seismic, raw_model

# ------------------------------------------------------------------------------
# 3. Motor de Tradução: Discreto para Contínuo (Dataset PyTorch)
# ------------------------------------------------------------------------------
class SeismicPointDataset(Dataset):
    def __init__(self, d_obs: np.ndarray):
        super().__init__()
        
        x_rec = np.arange(NUM_REC) * DX
        z_rec = np.array([10.0]) 
        t_vec = np.arange(NT) * DT
        
        T, Z, X = np.meshgrid(t_vec, z_rec, x_rec, indexing='ij')
        
        self.t_data = torch.tensor(T.flatten(), dtype=torch.float32)
        self.z_data = torch.tensor(Z.flatten(), dtype=torch.float32)
        self.x_data = torch.tensor(X.flatten(), dtype=torch.float32)
        
        u_reshaped = np.zeros((len(self.t_data), NUM_SHOTS))
        for i in range(NUM_SHOTS):
            u_reshaped[:, i] = d_obs[i, :, :].flatten()
            
        self.u_data = torch.tensor(u_reshaped, dtype=torch.float32)

        self.x_norm = self.normalize(self.x_data, 0.0, (NX-1)*DX)
        self.z_norm = self.normalize(self.z_data, 0.0, (NZ-1)*DX)
        self.t_norm = self.normalize(self.t_data, 0.0, (NT-1)*DT)

    def normalize(self, tensor: torch.Tensor, min_val: float, max_val: float) -> torch.Tensor:
        return 2.0 * ((tensor - min_val) / (max_val - min_val)) - 1.0

    def __len__(self) -> int:
        return len(self.x_data)

    def __getitem__(self, idx: int):
        return (self.x_norm[idx], self.z_norm[idx], self.t_norm[idx]), self.u_data[idx]

dataset_obs = SeismicPointDataset(seismic_obs)
print(f"[Data Engineering] Nuvem de pontos gerada com sucesso: {len(dataset_obs)} coordenadas.")

# ------------------------------------------------------------------------------
# 4. Inspeção Visual (O Modelo e a Nuvem de Pontos 3D)
# ------------------------------------------------------------------------------
print("[Visualização] Renderizando o Modelo Real e a Nuvem de Pontos (Receptores)...")

fig = plt.figure(figsize=(14, 6))

# Plot 1: O Modelo de Velocidade (Ground Truth)
ax1 = fig.add_subplot(1, 2, 1)
im1 = ax1.imshow(true_velocity, cmap='jet', aspect='auto', extent=[0, (NX-1)*DX, (NZ-1)*DX, 0])
ax1.set_title("Modelo Verdadeiro (Ground Truth)")
ax1.set_xlabel("Distância X (m)")
ax1.set_ylabel("Profundidade Z (m)")
fig.colorbar(im1, ax=ax1, label="Velocidade (m/s)")

# Plot 2: A Nuvem de Pontos 3D (X, Z, T)
# Para não travar o renderizador do matplotlib, plotamos 1 a cada 10 passos de tempo
ax2 = fig.add_subplot(1, 2, 2, projection='3d')

# Extraindo os dados desnormalizados para visualização física real
step = 10
x_plot = dataset_obs.x_data.numpy()[::step]
z_plot = dataset_obs.z_data.numpy()[::step]
t_plot = dataset_obs.t_data.numpy()[::step]

# Usando a amplitude do tiro central (índice 2) como mapa de cores
amp_plot = dataset_obs.u_data.numpy()[::step, 2] 

# Scatter plot 3D
sc = ax2.scatter(x_plot, t_plot, z_plot, c=amp_plot, cmap='seismic', s=5, alpha=0.8)
ax2.set_title("Nuvem de Pontos (Dados dos Receptores)")
ax2.set_xlabel("Distância X (m)")
ax2.set_ylabel("Tempo T (s)")
ax2.set_zlabel("Profundidade Z (m)")

# Ajustando os limites do eixo Z para mostrar que os pontos estão na superfície
ax2.set_zlim((NZ-1)*DX, 0)

fig.colorbar(sc, ax=ax2, label="Amplitude Acústica (Tiro Central)", pad=0.1)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# CELULA 3: ARQUITETURA DA REDE NEURAL (PINN PURA - MLP PADRÃO)
# ==============================================================================
# DIAGRAMA DE ARQUITETURA: PINN MULTI-SOURCE (BASELINE)
# ------------------------------------------------------------------------------
#  [ Entrada Contínua ]   [ Camadas Ocultas (6x128) ]     [ Saída (Amplitudes) ]
#  (Coordenadas Norm.)    (Ativação: Tanh - Suave)        (Pressão Acústica)
#
#       X_norm --+                                        +--> U_shot_1
#                |        +-----+   +-----+       +-----+ |--> U_shot_2
#       Z_norm --+------> | 128 |-->| 128 |-->...>| 128 |-+--> U_shot_3
#                |        +-----+   +-----+       +-----+ |--> U_shot_4
#       T_norm --+                                        +--> U_shot_5
#
#  *Obrigatório: Tanh garante derivadas de 2ª ordem não nulas (Laplaciano).*
# ==============================================================================

import torch
import torch.nn as nn

class PINN_Pura(nn.Module):
    """
    O QUE FAZ: Mapeia coordenadas contínuas (X, Z, T) diretamente para amplitudes sísmicas.
    
    PARA QUE SERVE: Atua como o baseline de Inteligência Artificial. Esta é a 
    arquitetura clássica (Multi-Layer Perceptron) sem nenhuma modificação. 
    
    HIPÓTESE CIENTÍFICA: Esperamos que esta rede sofra de 'Viés Espectral' (Spectral Bias), 
    tendo extrema dificuldade em aprender as altas frequências da onda sísmica, o que 
    provavelmente resultará em um gradiente geológico fraco ou ruidoso.
    """
    def __init__(self, in_features=3, out_features=5, hidden_layers=6, hidden_neurons=128):
        super().__init__() 
        self.layers = nn.ModuleList()
        
        # Camada de Entrada (Recebe estritamente 3 valores: X, Z, T)
        self.layers.append(nn.Linear(in_features, hidden_neurons))
        
        # Camadas Ocultas (A linha de montagem da extração de features)
        for _ in range(hidden_layers):
            self.layers.append(nn.Linear(hidden_neurons, hidden_neurons))
            
        # Camada de Saída (Prevê os 5 tiros simultaneamente)
        self.layers.append(nn.Linear(hidden_neurons, out_features))
        
    def forward(self, x_in: torch.Tensor, z_in: torch.Tensor, t_in: torch.Tensor) -> torch.Tensor:
        # Garante que as entradas sejam vetores coluna (Batch, 1)
        x_col = x_in.view(-1, 1)
        z_col = z_in.view(-1, 1)
        t_col = t_in.view(-1, 1)
        
        # Concatena as coordenadas no eixo das features
        u = torch.cat([x_col, z_col, t_col], dim=1)
        
        # Propagação pela MLP com ativação Tanh
        for i in range(len(self.layers) - 1):
            u = self.layers[i](u)
            u = torch.tanh(u)
            
        # Saída linear (sem ativação para não estrangular a amplitude acústica)
        u = self.layers[-1](u)
        return u

print("[Arquitetura] Instanciando a PINN Pura (Baseline MLP)...")
model_pinn = PINN_Pura(in_features=3, out_features=NUM_SHOTS, hidden_layers=6, hidden_neurons=128).to(device)
print(model_pinn)

In [ ]:
# ==============================================================================
# CELULA 4: MOTOR DA FISICA (AUTOGRAD + CORRECAO JACOBIANA)
# ==============================================================================
# DIAGRAMA DE FLUXO: PINN + Autograd + Regra da Cadeia
# ------------------------------------------------------------------------------
#  [ Predição da Rede (u_pred) ]
#          |
#          +---> (Autograd: Derivadas no Domínio Normalizado [-1, 1])
#          |      u_xx_norm, u_zz_norm, u_tt_norm
#          |
#          v
#  [ Correção Jacobiana (Regra da Cadeia) ]
#  Multiplica pelos fatores de escala (2 / (Max - Min))^2
#          |
#          v
#  [ Derivadas no Domínio Físico Real (Metros e Segundos) ]
#  u_xx_phys, u_zz_phys, u_tt_phys
#          |
#          v
#  [ Resíduo da PDE (Equação da Onda) ] -> u_tt - c^2 * (u_xx + u_zz) = 0
# ==============================================================================

from typing import Tuple

def get_gradient(output: torch.Tensor, input_var: torch.Tensor) -> torch.Tensor:
    """
    O QUE FAZ: Calcula a derivada parcial exata usando o Histórico de Operações (Autograd).
    PARA QUE SERVE: Extrair as taxas de variação da onda (inclinação, curvatura, aceleração)
    sem usar aproximações numéricas (como o FDTD faz).
    """
    grad_outputs = torch.ones_like(output)
    gradient = torch.autograd.grad(
        outputs=output,
        inputs=input_var,
        grad_outputs=grad_outputs,
        create_graph=True,
        retain_graph=True
    )[0] 
    return gradient

def compute_physics_loss(
    model: nn.Module, 
    x_norm: torch.Tensor, 
    z_norm: torch.Tensor, 
    t_norm: torch.Tensor, 
    c_velocity: torch.Tensor,
    scale_factors: dict
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    O QUE FAZ: Calcula o resíduo da Equação da Onda Acústica 2D, aplicando os 
    fatores de escala (Jacobianos) para reverter a normalização [-1, 1].
    PARA QUE SERVE: É o "Gabarito da Natureza". Força a rede neural a respeitar a 
    física. Se a rede inventar dados, esta função gera uma penalidade alta.
    """
    # Ativação do rastreamento de gradientes
    x_norm.requires_grad_(True)
    z_norm.requires_grad_(True)
    t_norm.requires_grad_(True)
    
    # Forward Pass: Predição do campo de onda (5 tiros simultâneos)
    u_pred = model(x_norm, z_norm, t_norm) 
    
    # 1. Derivadas no Domínio Normalizado (Autograd Puro)
    u_x_norm = get_gradient(u_pred, x_norm)
    u_xx_norm = get_gradient(u_x_norm, x_norm)
    
    u_z_norm = get_gradient(u_pred, z_norm)
    u_zz_norm = get_gradient(u_z_norm, z_norm)
    
    u_t_norm = get_gradient(u_pred, t_norm)
    u_tt_norm = get_gradient(u_t_norm, t_norm)
    
    # 2. Correção Jacobiana (Retorno ao Domínio Físico)
    j_x = scale_factors['x']  
    j_z = scale_factors['z']  
    j_t = scale_factors['t']  
    
    u_xx_phys = u_xx_norm * (j_x ** 2)
    u_zz_phys = u_zz_norm * (j_z ** 2)
    u_tt_phys = u_tt_norm * (j_t ** 2)
    
    # 3. O Teste da Natureza (Equação da Onda Acústica 2D)
    # Como u_pred tem shape [Batch, 5 Tiros], c_velocity [Batch] precisa ser 
    # expandido para [Batch, 1] para o broadcast correto da multiplicação.
    c_expanded = c_velocity.view(-1, 1)
    
    pde_residual = u_tt_phys - (c_expanded ** 2) * (u_xx_phys + u_zz_phys)
    
    # Erro Quadrático Médio da violação física
    loss_pde = torch.mean(pde_residual ** 2)
    
    return loss_pde, u_pred

print("[MLOps] Motor de Derivação Automática (Autograd) e PDE alocados em memória.")

In [ ]:
# ==============================================================================
# CELULA 5: O PRODUTO COMERCIAL (VELOCITY GRID)
# ==============================================================================
# DIAGRAMA DE INTERPOLAÇÃO GEOLÓGICA
# ------------------------------------------------------------------------------
#  [ Coordenadas Contínuas da PINN (x_norm, z_norm) ]
#                 |
#                 v
#  [ F.grid_sample (Interpolação Bilinear) ] <--- [ Matriz de Velocidade (NX, NZ) ]
#                 |                               (Parâmetro Treinável)
#                 v
#  [ Velocidade Exata no Ponto (c_velocity) ] ---> Injetada na Equação da Onda
# ==============================================================================

class VelocityGrid(nn.Module):
    """
    O QUE FAZ: Mantém a matriz de velocidades como parâmetros treináveis na GPU. 
    Utiliza interpolação bilinear (grid_sample) para fornecer a velocidade exata 
    em qualquer coordenada contínua (x, z) solicitada pela PINN.
    
    PARA QUE SERVE: Este é o PRODUTO COMERCIAL. Enquanto a PINN atua como o 
    simulador da onda, esta matriz é a geologia real sendo descoberta.
    """
    def __init__(self, nx: int, nz: int, initial_vel: float):
        super().__init__()
        # Formato exigido pelo grid_sample: [Batch, Canais, Altura, Largura]
        self.grid = nn.Parameter(torch.ones(1, 1, nz, nx) * initial_vel)
        
    def forward(self, x_norm: torch.Tensor, z_norm: torch.Tensor) -> torch.Tensor:
        # Agrupa as coordenadas [-1, 1] no formato [1, Batch, 1, 2] para o amostrador
        grid_coords = torch.cat([x_norm.unsqueeze(-1), z_norm.unsqueeze(-1)], dim=-1)
        grid_coords = grid_coords.view(1, -1, 1, 2)
        
        # Extrai a velocidade interpolada para cada ponto da nuvem
        c = F.grid_sample(self.grid, grid_coords, align_corners=True)
        return c.view(-1) # Retorna como vetor 1D

print("[Arquitetura] Instanciando o Grid de Velocidades (Produto Comercial)...")
vel_model = VelocityGrid(nx=NX, nz=NZ, initial_vel=1500.0).to(device)
print(f"Grid Shape: {vel_model.grid.shape}")

# ------------------------------------------------------------------------------
# Inspeção Visual do Modelo Inicial (O "Canvas" em Branco)
# ------------------------------------------------------------------------------
# Extraímos a matriz da GPU, removemos as dimensões extras (Batch, Canal) e passamos para NumPy
initial_grid = vel_model.grid.detach().cpu().squeeze().numpy()

plt.figure(figsize=(6, 5))
# Fixamos o vmin e vmax com os limites geológicos reais para vermos que estamos no "fundo" da escala
im = plt.imshow(initial_grid, cmap='jet', aspect='auto', vmin=1400, vmax=4500, extent=[0, (NX-1)*DX, (NZ-1)*DX, 0])
plt.colorbar(im, label="Velocidade (m/s)")
plt.title("Grid de Velocidade Inicial (Chute Cego - 1500 m/s)")
plt.xlabel("Distância X (m)")
plt.ylabel("Profundidade Z (m)")
plt.show()

In [ ]:
# ==============================================================================
# CELULA 6: ESTUDO DE ABLACAO - PINN PURA (OTIMIZACAO ESTRITA DE 1a ORDEM: ADAM)
# ==============================================================================
import time
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt

# Verificação de Hardware (Fail-Fast)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Sistema] Executando pipeline PINN. Dispositivo mapeado: {device}")
if device.type != 'cuda':
    print("[ALERTA CRITICO] CUDA não detectado. O treinamento de PINNs na CPU é inviável.")

print("[Ablacao] Iniciando Experimento 1: PINN Pura otimizada estritamente via ADAM...")

BATCH_SIZE = 8192
EPOCHS_ADAM = 100  # Reduzido para 100 (Fail-Fast otimizado)
LR_PINN = 1e-3
LR_VEL = 10.0
LAMBDA_DATA = 1.0
LAMBDA_PDE = 0.01

scale_factors = {'x': 2.0/((NX-1)*DX), 'z': 2.0/((NZ-1)*DX), 't': 2.0/((NT-1)*DT)}

model_pinn_adam = PINN_Pura(in_features=3, out_features=NUM_SHOTS, hidden_layers=6, hidden_neurons=128).to(device)
vel_model_adam = VelocityGrid(nx=NX, nz=NZ, initial_vel=1500.0).to(device)

dataloader = DataLoader(dataset_obs, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

optimizer_adam = torch.optim.Adam([
    {'params': model_pinn_adam.parameters(), 'lr': LR_PINN},
    {'params': vel_model_adam.parameters(), 'lr': LR_VEL}
])

scheduler = ReduceLROnPlateau(optimizer_adam, mode='min', factor=0.5, patience=10)

start_time = time.time()
for epoch in range(1, EPOCHS_ADAM + 1):
    model_pinn_adam.train()
    epoch_data_loss, epoch_pde_loss = 0.0, 0.0
    current_lr = optimizer_adam.param_groups[0]['lr']
    
    for coords, u_obs in dataloader:
        x_b, z_b, t_b = coords[0].to(device), coords[1].to(device), coords[2].to(device)
        u_obs = u_obs.to(device)
        
        x_c = (torch.rand(BATCH_SIZE, device=device) * 2.0) - 1.0
        z_c = (torch.rand(BATCH_SIZE, device=device) * 2.0) - 1.0
        t_c = (torch.rand(BATCH_SIZE, device=device) * 2.0) - 1.0
        
        optimizer_adam.zero_grad()
        
        c_batch = vel_model_adam(x_c, z_c)
        loss_pde, _ = compute_physics_loss(model_pinn_adam, x_c, z_c, t_c, c_batch, scale_factors)
        
        u_pred_data = model_pinn_adam(x_b, z_b, t_b)
        loss_data = torch.nn.MSELoss()(u_pred_data, u_obs)
        
        loss_total = (LAMBDA_DATA * loss_data) + (LAMBDA_PDE * loss_pde)
        loss_total.backward()
        
        torch.nn.utils.clip_grad_norm_(model_pinn_adam.parameters(), max_norm=1.0)
        optimizer_adam.step()
        
        with torch.no_grad():
            vel_model_adam.grid.clamp_(min=1400.0, max=4500.0)
            
        epoch_data_loss += loss_data.item()
        epoch_pde_loss += loss_pde.item()
        
    avg_data_loss = epoch_data_loss / len(dataloader)
    avg_pde_loss = epoch_pde_loss / len(dataloader)
    scheduler.step(avg_data_loss + avg_pde_loss)
    
    new_lr = optimizer_adam.param_groups[0]['lr']
    if new_lr < current_lr:
        print(f"[MLOps Alerta] Plato detectado na Epoca {epoch}. LR reduzida para {new_lr:.2e}")
    
    if epoch % 10 == 0 or epoch == 1:
        with torch.no_grad():
            v_max = vel_model_adam.grid.max().item()
        print(f"ADAM Only | Epoch [{epoch:03d}/{EPOCHS_ADAM}] | Data Loss: {avg_data_loss:.4e} | PDE Loss: {avg_pde_loss:.4e} | V_max: {v_max:.1f}")

print(f"[Ablacao] Experimento 1 concluido em {(time.time() - start_time)/60:.2f} minutos.")

inv_vel_adam = vel_model_adam.grid.detach().cpu().squeeze().numpy()
trace_inv_adam = inv_vel_adam[:, NX // 2]
depth_axis = np.arange(NZ) * DX

plt.figure(figsize=(6, 6))
plt.plot(true_velocity[:, NX // 2], depth_axis, 'k-', linewidth=2, label="Real (Ground Truth)")
plt.plot(trace_inv_adam, depth_axis, 'b--', linewidth=2, label="Apenas ADAM (100 Epocas)")
plt.gca().invert_yaxis()
plt.title("Perfil 1D - Estudo de Ablacao (Apenas ADAM)")
plt.xlabel("Velocidade Acustica (m/s)")
plt.ylabel("Profundidade (m)")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.show()

### 🔬 ANÁLISE FORENSE (Experimento 1): O Colapso por Viés Espectral

O resultado do Experimento 1 (Apenas ADAM) comprova a falha da arquitetura PINN Pura (Vanilla MLP) em resolver problemas inversos de propagação de onda. A análise da telemetria revela um fenômeno matemático conhecido como **Viés Espectral (Spectral Bias)**.

**O Diagnóstico Físico-Matemático:**
1. **A "Trapaça" da Rede Neural:** Observamos que a `PDE Loss` (o erro da física) caiu drasticamente, enquanto a `Data Loss` e a velocidade máxima (`V_max`) permaneceram estagnadas. Redes neurais densas possuem uma aversão natural a aprender funções de alta frequência. Para minimizar o erro da Equação da Onda sem o esforço de mapear a complexa cinemática sísmica, a rede convergiu para uma solução trivial: ela previu um campo de onda "plano" (quase nulo).
2. **A Morte do Gradiente Geológico:** A atualização da matriz de velocidades depende da Regra da Cadeia, sendo o gradiente diretamente proporcional à curvatura espacial da onda $(u_{xx} + u_{zz})$. Como a rede previu uma onda plana (curvatura nula), o gradiente transmitido à geologia foi zero.
3. **A Cegueira do Otimizador:** O ADAM, sendo um otimizador estrito de primeira ordem, depende exclusivamente desse gradiente para se mover. Sem gradiente, a taxa de aprendizado colapsou (atingindo a casa de $10^{-6}$) e a matriz de velocidades permaneceu inalterada (linha azul vertical no Perfil 1D).

**Conclusão:** A otimização de primeira ordem sobre uma PINN Pura é incapaz de superar o Viés Espectral. A rede neural "desliga" a física antes que a geologia possa ser atualizada.

In [ ]:
# ==============================================================================
# CELULA 7: ESTUDO DE ABLACAO - PINN PURA (OTIMIZACAO ESTRITA DE 2a ORDEM: L-BFGS)
# ==============================================================================
# Verificação de Hardware (Fail-Fast)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Sistema] Executando pipeline PINN. Dispositivo mapeado: {device}")
if device.type != 'cuda':
    print("[ALERTA CRITICO] CUDA não detectado. O treinamento de PINNs na CPU é inviável.")

print("[Ablacao] Iniciando Experimento 2: PINN Pura otimizada estritamente via L-BFGS...")

EPOCHS_LBFGS = 100  # Reduzido para 100 (Fail-Fast otimizado)

model_pinn_lbfgs = PINN_Pura(in_features=3, out_features=NUM_SHOTS, hidden_layers=6, hidden_neurons=128).to(device)
vel_model_lbfgs = VelocityGrid(nx=NX, nz=NZ, initial_vel=1500.0).to(device)

optimizer_lbfgs = torch.optim.LBFGS(
    list(model_pinn_lbfgs.parameters()) + list(vel_model_lbfgs.parameters()),
    lr=1.0, 
    max_iter=20, 
    max_eval=25, 
    tolerance_grad=1e-7, 
    tolerance_change=1e-9,
    history_size=50, 
    line_search_fn="strong_wolfe" 
)

lbfgs_iterator = iter(DataLoader(dataset_obs, batch_size=16384, shuffle=True))
coords_lbfgs, u_obs_lbfgs = next(lbfgs_iterator)
x_l, z_l, t_l = coords_lbfgs[0].to(device), coords_lbfgs[1].to(device), coords_lbfgs[2].to(device)
u_obs_lbfgs = u_obs_lbfgs.to(device)

x_c_l = (torch.rand(16384, device=device) * 2.0) - 1.0
z_c_l = (torch.rand(16384, device=device) * 2.0) - 1.0
t_c_l = (torch.rand(16384, device=device) * 2.0) - 1.0

start_time = time.time()
for epoch in range(1, EPOCHS_LBFGS + 1):
    
    def closure():
        optimizer_lbfgs.zero_grad()
        
        c_batch_l = vel_model_lbfgs(x_c_l, z_c_l)
        loss_pde_l, _ = compute_physics_loss(model_pinn_lbfgs, x_c_l, z_c_l, t_c_l, c_batch_l, scale_factors)
        
        u_pred_data_l = model_pinn_lbfgs(x_l, z_l, t_l)
        loss_data_l = torch.nn.MSELoss()(u_pred_data_l, u_obs_lbfgs)
        
        loss_total_l = (LAMBDA_DATA * loss_data_l) + (LAMBDA_PDE * loss_pde_l)
        loss_total_l.backward()
        
        torch.nn.utils.clip_grad_norm_(model_pinn_lbfgs.parameters(), max_norm=1.0)
        return loss_total_l

    optimizer_lbfgs.step(closure)
    
    if epoch % 10 == 0 or epoch == 1:
        loss_val = closure().item()
        with torch.no_grad():
            vel_model_lbfgs.grid.clamp_(min=1400.0, max=4500.0)
            v_max = vel_model_lbfgs.grid.max().item()
        print(f"L-BFGS Only | Epoch [{epoch:03d}/{EPOCHS_LBFGS}] | Total Loss: {loss_val:.4e} | V_max: {v_max:.1f}")

print(f"[Ablacao] Experimento 2 concluido em {(time.time() - start_time)/60:.2f} minutos.")

inv_vel_lbfgs = vel_model_lbfgs.grid.detach().cpu().squeeze().numpy()
trace_inv_lbfgs = inv_vel_lbfgs[:, NX // 2]

plt.figure(figsize=(6, 6))
plt.plot(true_velocity[:, NX // 2], depth_axis, 'k-', linewidth=2, label="Real (Ground Truth)")
plt.plot(trace_inv_lbfgs, depth_axis, 'r--', linewidth=2, label="Apenas L-BFGS (100 Epocas)")
plt.gca().invert_yaxis()
plt.title("Perfil 1D - Estudo de Ablacao (Apenas L-BFGS)")
plt.xlabel("Velocidade Acustica (m/s)")
plt.ylabel("Profundidade (m)")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.show()

### 🔬 ANÁLISE FORENSE (Experimento 2): O Colapso da Busca Linear (L-BFGS)

O resultado do Experimento 2 (Apenas L-BFGS) comprova a inviabilidade de aplicar otimizadores de segunda ordem diretamente sobre redes neurais não inicializadas (pesos aleatórios). O treinamento estagnou computacionalmente logo na primeira época.

**O Diagnóstico Físico-Matemático:**
1. **Aproximação Hessiana sobre o Caos:** O L-BFGS é um método Quase-Newton que estima a curvatura do espaço de parâmetros. Ao ser aplicado sobre uma rede neural recém-instanciada, a topologia da função de perda é altamente não-convexa e caótica. A matriz Hessiana calculada neste estado aponta para direções de atualização espúrias.
2. **Falha na Condição de *Strong Wolfe*:** Para garantir a estabilidade, o L-BFGS utiliza um algoritmo de busca linear (*Line Search*) que exige uma redução suficiente na função de perda. Como as direções propostas pela Hessiana corrompida são inválidas, a busca linear entra em um loop exaustivo de reavaliações da função `closure()`, calculando derivadas de segunda ordem milhares de vezes sem conseguir dar um passo válido.
3. **Custo Computacional Inviável:** O tempo de processamento torna-se proibitivo, caracterizando uma falha algorítmica antes mesmo de qualquer atualização geológica ocorrer.

**Conclusão do Estudo de Ablação:**
A otimização de PINNs para Inversão FWI exige estritamente uma **Arquitetura Híbrida (Adam $\to$ L-BFGS)**. O Adam é matematicamente necessário para o pré-treinamento (retirando os pesos do estado caótico inicial), criando um espaço localmente convexo onde o L-BFGS pode operar com segurança para esculpir as altas frequências geológicas.

# FUNDAMENTAÇÃO TEÓRICA: A Arquitetura "Fourier + Híbrida"

Após comprovarmos a falha da PINN Pura (Vanilla), a literatura de ponta em Inteligência Artificial aplicada à Física (SciML) nos direciona para uma arquitetura composta por duas inovações simultâneas: Otimização Híbrida e Fourier Features. Abaixo, desconstruímos o porquê dessa necessidade.

## 1. O Problema de Otimização: Por que a Abordagem Híbrida (Adam -> L-BFGS)?
O treinamento de uma rede neural é a busca pelo ponto mais baixo em um terreno montanhoso e caótico (a função de perda).

* **O papel do ADAM (O Trator):** Quando a rede nasce, seus pesos são aleatórios. O terreno é extremamente acidentado. O Adam é um otimizador de primeira ordem robusto. Ele desce a montanha rapidamente, ignorando pequenos buracos. Contudo, quando ele chega em um vale muito largo e quase plano (que na geofísica representa as camadas profundas, onde o gradiente é fraco), ele acha que o trabalho acabou e para.
* **O papel do L-BFGS (O Bisturi):** O L-BFGS é um otimizador de segunda ordem. Ele calcula a curvatura do terreno (Matriz Hessiana). Se o colocarmos no topo da montanha caótica (como fizemos no Experimento 2), ele tenta calcular a curvatura de cada pedra, entra em colapso matemático e trava. Mas, se o acionarmos *depois* que o Adam já nos levou para o vale plano, o L-BFGS usa a curvatura para encontrar o fundo exato do vale em poucos passos, esculpindo a geologia profunda.

**Conclusão 1:** Precisamos do Adam para organizar o caos inicial e do L-BFGS para o refinamento geológico profundo.

## 2. O Problema de Percepção: Por que Fourier Features?
Mesmo que o otimizador consiga caminhar perfeitamente, ele não pode consertar o que a rede neural não consegue enxergar.

* **O Viés Espectral (A Miopia da Rede):** Matematicamente, redes neurais densas (MLPs) aprendem funções suaves e de baixa frequência muito rápido. Elas são "míopes". Uma interface geológica (o salto abrupto de 1500 m/s para 3200 m/s) é um evento de alta frequência. Se entregarmos apenas as coordenadas puras $(X, Z, T)$ para a rede, ela vai borrar essa interface, gerando uma onda plana e matando o gradiente (como vimos no Experimento 1).
* **A Solução (Os Óculos de Grau):** As *Fourier Features* (ou Positional Encoding) atuam como lentes corretivas. Antes de entregar as coordenadas $(X, Z, T)$ para a rede, nós as passamos por uma camada de funções trigonométricas (senos e cossenos) multiplicadas por frequências aleatórias. 

**A Matemática da Percepção:**
Em vez de a rede receber um simples $X = 10$, ela recebe um vetor de altas frequências: $[\sin(2\pi f_1 X), \cos(2\pi f_1 X), \sin(2\pi f_2 X), ...]$. 
Isso força a rede neural a prestar atenção nas variações abruptas do espaço. Ela deixa de ser míope e passa a enxergar as bordas nítidas da propagação da onda sísmica. Se a rede enxerga a onda nítida, a Equação da Onda gera um resíduo forte, e o gradiente flui para a matriz de velocidades.

**Síntese da Arquitetura Final:**
As *Fourier Features* garantem que a rede enxergue a física correta (gerando o gradiente), enquanto a *Otimização Híbrida* garante que esse gradiente seja usado de forma eficiente para atualizar a rocha desde a superfície até o fundo do modelo.